# 02 - Data Preparation
Tujuan: membersihkan data, melakukan feature engineering, encoding,
dan membagi data menjadi train-test set yang siap dipakai untuk modeling.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape awal:", df.shape)

Shape awal: (7043, 21)


## 1. Cleaning
- Drop customerID (tidak prediktif, hanya identifier)
- Fix TotalCharges (object -> numeric)

In [3]:
df = df.drop(columns=["customerID"])

In [4]:
# TotalCharges punya beberapa baris berisi string kosong " " (customer baru, tenure = 0)
# pd.to_numeric dengan errors="coerce" akan mengubah string kosong jadi NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Jumlah NaN di TotalCharges setelah konversi:", df["TotalCharges"].isnull().sum())

# Untuk customer dengan tenure = 0 (baru join), TotalCharges yang masuk akal adalah 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Jumlah NaN setelah fillna:", df["TotalCharges"].isnull().sum())

Jumlah NaN di TotalCharges setelah konversi: 11
Jumlah NaN setelah fillna: 0


## 2. Feature Engineering
- tenure_group: binning tenure ke beberapa kategori
- num_services: total jumlah layanan tambahan yang di-subscribe

In [5]:
def tenure_to_group(tenure):
    if tenure <= 12:
        return "0-12"
    elif tenure <= 24:
        return "13-24"
    elif tenure <= 48:
        return "25-48"
    elif tenure <= 60:
        return "49-60"
    else:
        return "61+"

df["tenure_group"] = df["tenure"].apply(tenure_to_group)

print(df["tenure_group"].value_counts())

0-12     2186
25-48    1594
61+      1407
13-24    1024
49-60     832
Name: tenure_group, dtype: int64


In [6]:
# Hitung berapa banyak layanan tambahan (selain phone & internet dasar)
# yang di-subscribe oleh customer. Nilai "Yes" dihitung 1, selain itu 0.
service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

print(df["num_services"].value_counts().sort_index())

0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: num_services, dtype: int64


## 3. Encoding
- Binary categorical (Yes/No, Male/Female, dst) -> 0/1
- Multi-category categorical -> one-hot encoding
- Target (Churn) -> 0/1

In [7]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(df["Churn"].value_counts())

0    5174
1    1869
Name: Churn, dtype: int64


In [8]:
binary_cols = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]

for col in binary_cols:
    print(f"{col}: {df[col].unique()}")

# gender: Male/Female -> 1/0
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})

# kolom Yes/No lainnya -> 1/0
for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
    df[col] = df[col].map({"Yes": 1, "No": 0})

gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
PaperlessBilling: ['Yes' 'No']


In [9]:
# drop_first=True untuk mengurangi redundansi (menghindari dummy variable trap)
categorical_cols = [
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaymentMethod", "tenure_group"
]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Shape setelah encoding:", df_encoded.shape)
df_encoded.head()

Shape setelah encoding: (7043, 36)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_13-24,tenure_group_25-48,tenure_group_49-60,tenure_group_61+
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,1,0,0,0,0,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,1,0,0,0,1,0,1,0,0
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,1,0,0,0,0
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,0,1,0,0,0,0,0,1,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,1,0,0,0,0,0


## 4. Train-Validation-Test Split
- 3-way split: Train 64% / Validation 16% / Test 20%
- stratify=y agar proporsi churn/non-churn tetap konsisten di semua split

PERBAIKAN vs versi sebelumnya: sebelumnya hanya ada train/test (80:20).
Test set dipakai untuk MEMILIH threshold & model terbaik, sekaligus untuk
MELAPORKAN performa akhir -> ini "double-dipping" yang membuat metrik
hasil akhir bias optimis (test set jadi tidak lagi benar-benar "unseen").

Solusi: tambahkan validation set. Val dipakai khusus untuk memilih
threshold optimal & model terbaik. Test set HANYA disentuh sekali,
di akhir, untuk melaporkan performa final yang jujur.

In [10]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["Churn"])
y = df_encoded["Churn"]

# Split 1: pisahkan test set (20%) di awal - tidak disentuh sampai evaluasi akhir
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split 2: dari sisa 80%, pisahkan validation set (20% dari sisa = 16% dari total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp
)

print("Train shape:     ", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape :     ", X_test.shape)
print("\nProporsi churn di train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nProporsi churn di validation:")
print(y_val.value_counts(normalize=True).round(3))
print("\nProporsi churn di test:")
print(y_test.value_counts(normalize=True).round(3))

Train shape:      (4507, 35)
Validation shape: (1127, 35)
Test shape :      (1409, 35)

Proporsi churn di train:
0    0.735
1    0.265
Name: Churn, dtype: float64

Proporsi churn di validation:
0    0.735
1    0.265
Name: Churn, dtype: float64

Proporsi churn di test:
0    0.735
1    0.265
Name: Churn, dtype: float64


## 5. Simpan hasil ke file (agar bisa langsung dipakai di script modeling)

In [11]:
X_train.to_csv("data/X_train.csv", index=False)
X_val.to_csv("data/X_val.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_val.to_csv("data/y_val.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)

print("Data preparation selesai. File tersimpan di folder data/:")
print("- X_train.csv, X_val.csv, X_test.csv, y_train.csv, y_val.csv, y_test.csv")

Data preparation selesai. File tersimpan di folder data/:
- X_train.csv, X_val.csv, X_test.csv, y_train.csv, y_val.csv, y_test.csv


## Ringkasan
1. customerID di-drop, TotalCharges dikonversi ke numerik (NaN -> 0 untuk tenure=0).
2. Fitur baru: tenure_group (binning) dan num_services (jumlah layanan tambahan).
3. Encoding: binary -> 0/1, multi-category -> one-hot (drop_first=True).
4. Train-test split 80:20 dengan stratify pada target.
5. Data siap pakai disimpan ke data/X_train.csv, X_test.csv, y_train.csv, y_test.csv.

Lanjut ke: 03_modeling.py